# 🚀 Artemis VLM Pipeline: Complete Walkthrough

This notebook demonstrates the full pipeline:
1. **Load sample from SQL** (vlm_sample table)
2. **Router prediction** (which VLM is best?)
3. **Load Balancer scheduling** (SLA/capacity aware)
4. **Inference** (actual VLM call)

```
SQL Sample → Router → Load Balancer → Inference → Response
```

## 1️⃣ Setup: Path Configuration

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# === Path Setup ===
import sys
from pathlib import Path

# Notebook paths
NOTEBOOK_DIR = Path.cwd()
ARTEMIS_DIR = NOTEBOOK_DIR.parent.parent  # artemis_final/
ROOT_DIR = ARTEMIS_DIR.parent             # Which_VLM_Router/
CHECKPOINTS_DIR = ARTEMIS_DIR / 'checkpoints'

# Add to sys.path for imports
for p in [str(ARTEMIS_DIR), str(ROOT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"📁 ROOT_DIR: {ROOT_DIR}")

import time



print(f'Checkpoints: {CHECKPOINTS_DIR}')
print('✓ Paths configured')

📁 ROOT_DIR: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router
Checkpoints: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/artemis_final/checkpoints
✓ Paths configured


## 2️⃣ Load Sample from SQL Database

In [3]:
from sqlalchemy import create_engine, text
import base64
from io import BytesIO

# Import DB config from ares
from artemis_final.ares.configs.db_config import DB_URL, TABLES

def load_random_sample_with_image():
    """Load a random sample WITH image from database."""
    try:
        engine = create_engine(DB_URL)
        
        with engine.connect() as conn:
            # Join vlm_samples with vlm_images to get sample + image together
            result = conn.execute(text(f"""
                SELECT 
                    s.sample_id, 
                    s.prompt_text,
                    s.router_task, 
                    s.source_dataset,
                    s.ground_truth,
                    i.image_bytes,
                    i.img_width,
                    i.img_height
                FROM {TABLES['samples']} s
                LEFT JOIN {TABLES['images']} i ON s.image_id = i.image_id
                WHERE s.prompt_text IS NOT NULL
                  AND i.image_bytes IS NOT NULL
                ORDER BY RANDOM()
                LIMIT 1
            """))
            row = result.fetchone()
        
        if row:
            return {
                'sample_id': row[0],
                'prompt': row[1],
                'router_task': row[2] or 'vqa',
                'source_dataset': row[3] or 'unknown',
                'ground_truth': row[4],
                'image_bytes': row[5],  # Binary image data
                'img_width': row[6],
                'img_height': row[7],
            }
        return None
    except Exception as e:
        print(f'⚠️ Database error: {e}')
        return None

# Try loading from SQL
sample = load_random_sample_with_image()

if sample is None:
    sample = {
        'sample_id': 'synthetic_001',
        'prompt': 'Extract all text from this receipt image.',
        'router_task': 'ocr',
        'source_dataset': 'synthetic',
        'ground_truth': None,
        'image_bytes': None,
        'img_width': None,
        'img_height': None,
    }
    print('⚠️ Using synthetic sample (no DB connection or no samples with images)')
else:
    print('✓ Loaded sample WITH image from SQL')

print(f'\n📋 Sample:')
print(f'   ID: {sample["sample_id"]}')
print(f'   Task: {sample["router_task"]}')
print(f'   Dataset: {sample["source_dataset"]}')
print(f'   Prompt: {sample["prompt"][:80]}...')
if sample['image_bytes']:
    print(f'   Image: {sample["img_width"]}x{sample["img_height"]} ({len(sample["image_bytes"])/1024:.1f} KB)')
else:
    print('   Image: None')

✓ Loaded sample WITH image from SQL

📋 Sample:
   ID: intergps_106_2a572317
   Task: geometry_reasoning
   Dataset: cauldron
   Prompt: Question: The segment is tangent to the circle. Find x.
Choices:
A. 7
B. 8
C. 9
...
   Image: 582x492 (48.2 KB)


## 3️⃣ Router: Predict Best Model

In [4]:
import torch
from artemis_final.router.artemis_router import ClassicalRouterInference
if torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f'🚀 Using CUDA: {torch.cuda.get_device_name(0)}')
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
    print('🍎 Using Apple MPS')
else:
    DEVICE = 'cpu'
    print('💻 Using CPU')
# Load router
router = ClassicalRouterInference(
    checkpoint_path=str(CHECKPOINTS_DIR / 'best_classical_router.pt'),
    device=DEVICE,
    verbose=True
)

print(f'\n📊 Available models: {router.model_names}')
print(f'📊 Available modes: {router.mode_names}')

🍎 Using Apple MPS
[INFO] Loading ClassicalRouter from: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/artemis_final/checkpoints/best_classical_router.pt
[INFO] Model loaded on device: mps
[INFO] Models: ['deepseek_ocr', 'gemma_3_27b', 'qwen2_5_vl_3b', 'qwen2_5_vl_7b', 'qwen3_vl_8b_thinking']
[INFO] Modes: ['accuracy', 'balanced', 'cheap', 'fast']

📊 Available models: ['deepseek_ocr', 'gemma_3_27b', 'qwen2_5_vl_3b', 'qwen2_5_vl_7b', 'qwen3_vl_8b_thinking']
📊 Available modes: ['accuracy', 'balanced', 'cheap', 'fast']


In [5]:
# Route the sample
router_result = router.route(
    prompt=sample['prompt'],
    mode='balanced',  # or 'accuracy', 'cheap', 'fast'
    metadata={
        'router_task': sample['router_task'],
        'source_dataset': sample.get('source_dataset', 'unknown')
    }
)

# Get router confidence (max probability = confidence in chosen model)
max_prob = max(router_result['probs'].values())

print('🔮 Router Prediction:')
print(f'   Chosen model: {router_result["chosen_model"]}')
print(f'   Router confidence: {max_prob:.3f}')  # Max probability = confidence
print(f'   Inference time: {router_result["inference_ms"]:.1f}ms')
print(f'   Probabilities:')
for model, prob in sorted(router_result['probs'].items(), key=lambda x: -x[1]):
    marker = '→' if model == router_result['chosen_model'] else ' '
    print(f'     {marker} {model}: {prob:.3f}')

🔮 Router Prediction:
   Chosen model: gemma_3_27b
   Router confidence: 0.413
   Inference time: 115.9ms
   Probabilities:
     → gemma_3_27b: 0.413
       qwen2_5_vl_3b: 0.277
       qwen2_5_vl_7b: 0.199
       qwen3_vl_8b_thinking: 0.081
       deepseek_ocr: 0.029


## 4️⃣ Load Balancer: SLA-Aware Scheduling

In [6]:
from artemis_final.load_balancer.public_api import (
    ArtemisLoadBalancer,
    RouterOutput,
    SchedulingContext,
    StatsRegistry,
    load_capacity_config,
)

# Load model capacity configs
model_configs = load_capacity_config()

print('📊 Loaded model configs:')
for name, cfg in model_configs.items():
    print(f'   {name}: latency={cfg.base_latency_ms}ms, SLA={cfg.sla_ms}ms')

📊 Loaded model configs:
   deepseek_ocr: latency=300ms, SLA=1500ms
   qwen2_5_vl_3b: latency=400ms, SLA=1500ms
   qwen2_5_vl_7b: latency=800ms, SLA=2500ms
   qwen3_vl_8b_thinking: latency=1200ms, SLA=3000ms
   gemma_3_27b: latency=2000ms, SLA=4000ms


In [7]:
# Create stats registry (synthetic for now - replace with real stats from ares)
synthetic_stats = {
    'ocr': {
        'deepseek_ocr': {'avg_latency_ms': 300, 'avg_accuracy': 0.95},
        'qwen2_5_vl_3b': {'avg_latency_ms': 400, 'avg_accuracy': 0.88},
        'qwen2_5_vl_7b': {'avg_latency_ms': 800, 'avg_accuracy': 0.92},
        'qwen3_vl_8b_thinking': {'avg_latency_ms': 1200, 'avg_accuracy': 0.90},
        'gemma_3_27b': {'avg_latency_ms': 2000, 'avg_accuracy': 0.85},
    },
    'vqa': {
        'deepseek_ocr': {'avg_latency_ms': 350, 'avg_accuracy': 0.70},
        'qwen2_5_vl_3b': {'avg_latency_ms': 450, 'avg_accuracy': 0.82},
        'qwen2_5_vl_7b': {'avg_latency_ms': 900, 'avg_accuracy': 0.88},
        'qwen3_vl_8b_thinking': {'avg_latency_ms': 1400, 'avg_accuracy': 0.91},
        'gemma_3_27b': {'avg_latency_ms': 2200, 'avg_accuracy': 0.93},
    },
    # Add more task types as needed
}

stats_registry = StatsRegistry(synthetic_stats)

# Create load balancer
lb = ArtemisLoadBalancer(
    model_configs=model_configs,
    stats_registry=stats_registry,
    latency_sla_ms={'default': 2000.0},
    max_accuracy_drop=0.05,
    scheduling_mode='capacity_aware'
)

print('✓ Load Balancer created')

✓ Load Balancer created


In [8]:
# Convert router output to load balancer format
max_prob = max(router_result['probs'].values()) if router_result['probs'] else 0.0
    
lb_input = RouterOutput(
    sample_id=sample['sample_id'],
    task_type=sample['router_task'],
    router_probs=router_result['probs'],
    preferred_model=router_result['chosen_model'],
        max_prob=max_prob
)

context = SchedulingContext(
    arrival_ts_ms=time.time() * 1000,
    load_profile='medium',
    metadata={'router_latency_ms': router_result['inference_ms']}
)

# Schedule
decision = lb.schedule(lb_input, context)

print('📋 Load Balancer Decision:')
print(f'   Router preferred: {decision.preferred_model}')
print(f'   LB chose: {decision.chosen_model}')
print(f'   Same model? {decision.preferred_model == decision.chosen_model}')
print(f'   Predicted latency: {decision.total_latency_ms:.1f}ms')
print(f'   Queue delay: {decision.queue_delay_ms:.1f}ms')
print(f'   SLA violated: {decision.sla_violated}')

Missing 'avg_accuracy' stats for task='geometry_reasoning', model='gemma_3_27b'. Using default value.
Missing 'avg_latency_ms' stats for task='geometry_reasoning', model='gemma_3_27b'. Using default value.
Missing 'cost_per_request_usd' stats for task='geometry_reasoning', model='gemma_3_27b'. Using default value.
Missing 'avg_latency_ms' stats for task='geometry_reasoning', model='qwen2_5_vl_3b'. Using default value.
Missing 'cost_per_request_usd' stats for task='geometry_reasoning', model='qwen2_5_vl_3b'. Using default value.
Missing 'avg_accuracy' stats for task='geometry_reasoning', model='qwen2_5_vl_3b'. Using default value.
Missing 'avg_latency_ms' stats for task='geometry_reasoning', model='qwen2_5_vl_7b'. Using default value.
Missing 'cost_per_request_usd' stats for task='geometry_reasoning', model='qwen2_5_vl_7b'. Using default value.
Missing 'avg_accuracy' stats for task='geometry_reasoning', model='qwen2_5_vl_7b'. Using default value.


📋 Load Balancer Decision:
   Router preferred: gemma_3_27b
   LB chose: gemma_3_27b
   Same model? True
   Predicted latency: 1000.0ms
   Queue delay: 0.0ms
   SLA violated: False


## 5️⃣ Inference: Call the VLM

In [9]:
# Check if inference client is available
try:
    # Use inference_engine module (now populated)
    from artemis_final.inference_engine import WhichVLMClient
    INFERENCE_AVAILABLE = True
    print('✓ WhichVLMClient available (from inference_engine)')
except ImportError as e:
    try:
        # Fallback to ares
        from artemis_final.ares.inference_api_call.client import WhichVLMClient
        INFERENCE_AVAILABLE = True
        print('✓ WhichVLMClient available (from ares)')
    except ImportError as e2:
        INFERENCE_AVAILABLE = False
        print(f'⚠️ Inference client not available: {e2}')

✓ WhichVLMClient available (from inference_engine)


In [10]:
# Load inference client from config
if INFERENCE_AVAILABLE:
    models_yaml = ROOT_DIR / 'artemis_final' / 'ares' / 'configs' / 'models.yaml'
    
    if models_yaml.exists():
        try:
            client = WhichVLMClient.from_yaml(str(models_yaml))
            print('✓ Loaded inference client')
            print(f'   VLM models: {client.list_vlm_models()}')
        except Exception as e:
            print(f'⚠️ Could not load client: {e}')
            client = None
    else:
        print(f'⚠️ Config not found: {models_yaml}')
        client = None
else:
    client = None

✓ Loaded inference client
   VLM models: ['gemma_3_27b', 'gemma_3_27b', 'qwen3_vl_8b_thinking', 'qwen3_vl_8b_thinking', 'qwen2_5_vl_7b', 'qwen2_5_vl_7b', 'qwen2_5_vl_3b', 'qwen2_5_vl_3b', 'deepseek_ocr', 'deepseek_ocr']


In [11]:
import math

def compute_confidence_from_logprobs(logprobs_data):
    """
    Compute confidence score from logprobs.
    Uses average token probability as confidence proxy.
    """
    if not logprobs_data:
        return None, "no_logprobs"
    
    content = logprobs_data.get("content", [])
    if not content:
        return None, "empty_content"
    
    # Extract logprobs for each token
    token_probs = []
    for token_info in content:
        if isinstance(token_info, dict):
            logprob = token_info.get("logprob")
            if logprob is not None:
                prob = math.exp(logprob)  # Convert log prob to probability
                token_probs.append(prob)
    
    if not token_probs:
        return None, "no_valid_tokens"
    
    # Confidence = geometric mean of token probabilities (or use first N tokens)
    # For simplicity, use average of first 10 tokens
    first_n = token_probs[:10]
    avg_prob = sum(first_n) / len(first_n)
    
    return avg_prob, "logprobs"

# Execute VLM inference with image
if client is not None:
    chosen_model = decision.chosen_model
    
    print(f'🚀 Calling VLM: {chosen_model}')
    print(f'   Prompt: {sample["prompt"][:60]}...')
    
    try:
        if sample['image_bytes']:
            # Convert bytes to PIL Image for the VLM client
            from PIL import Image
            image = Image.open(BytesIO(sample['image_bytes']))
            
            print(f'   Image: {image.width}x{image.height}')
            
            # Call VLM with image (request logprobs)
            result = client.vlm.run_image(
                image=image,
                text=sample['prompt'],
                models=[chosen_model],
                max_tokens=500,
                logprobs=True,
                top_logprobs=1
            )
            
            model_result = result.get(chosen_model, {})
            if model_result.get('ok'):
                print(f'\n✅ Response from {chosen_model}:')
                print(f'   Latency: {model_result["latency_ms"]:.0f}ms')
                
                # Compute confidence from logprobs
                logprobs_data = model_result.get('logprobs')
                confidence, source = compute_confidence_from_logprobs(logprobs_data)
                
                if confidence is not None:
                    print(f'   VLM Confidence: {confidence:.3f} (from {source})')
                else:
                    print(f'   VLM Confidence: N/A ({source})')
                
                # Token usage
                usage = model_result.get('usage', {})
                if usage:
                    print(f'   Tokens: {usage.get("prompt_tokens", "?")} in, {usage.get("completion_tokens", "?")} out')
                
                # Response text
                print(f'\n   📝 Response:')
                print(f'   {model_result["response_text"][:600]}')
                
                if sample.get('ground_truth'):
                    print(f'\n   📌 Ground Truth:')
                    print(f'   {sample["ground_truth"][:300]}')
            else:
                print(f'❌ Error: {model_result.get("error")}')
        else:
            print('⚠️ No image data - skipping VLM inference')
            print(f'   Would have called: {chosen_model}')
            
    except Exception as e:
        print(f'❌ Inference error: {e}')
        import traceback
        traceback.print_exc()
else:
    print('⚠️ Inference client not available')
    print(f'   Would have called: {decision.chosen_model} with image')

🚀 Calling VLM: gemma_3_27b
   Prompt: Question: The segment is tangent to the circle. Find x.
Choi...
   Image: 582x492
❌ Error: None


## 📊 Pipeline Summary

| Step | Component | Output |
|------|-----------|--------|
| 1 | SQL | Sample with prompt + task |
| 2 | Router | Model probabilities + chosen model |
| 3 | Load Balancer | Final model (may differ) + SLA check |
| 4 | Inference | Actual VLM response |

In [12]:
# Print full pipeline summary
print('='*60)
print('PIPELINE SUMMARY')
print('='*60)
print(f'Sample ID: {sample["sample_id"]}')
print(f'Task Type: {sample["router_task"]}')
print(f'Prompt: {sample["prompt"][:60]}...')
print('-'*60)
print(f'Router chose: {router_result["chosen_model"]} (latency: {router_result["inference_ms"]:.1f}ms)')
print(f'Load Balancer chose: {decision.chosen_model}')
print(f'Predicted total latency: {decision.total_latency_ms:.1f}ms')
print(f'SLA violated: {decision.sla_violated}')
print('='*60)

PIPELINE SUMMARY
Sample ID: intergps_106_2a572317
Task Type: geometry_reasoning
Prompt: Question: The segment is tangent to the circle. Find x.
Choi...
------------------------------------------------------------
Router chose: gemma_3_27b (latency: 115.9ms)
Load Balancer chose: gemma_3_27b
Predicted total latency: 1000.0ms
SLA violated: False
